### Document Data

Uploads pre-generated PDFs and loads structured metadata tables for every
document type used by the operational dashboard demo:

- **Food safety inspections** — inspection reports per ghost kitchen location
- **Menus** — brand menu PDFs with nutritional info and allergens
- **Legal complaints** — employment, liability, vendor disputes
- **Regulatory documents** — health permits, fire safety, zoning, FDA
- **Audit reports** — financial, operational, food safety, supply chain
- **Consultancy reports** — strategy, operations, AI transformation, workforce

PDFs are committed in the repo (`data/<topic>/pdfs/`) and are the source of
truth. Regenerate manually via the `generate_*.py` scripts when content needs
to change.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create catalog, schemas, and volumes for all document types

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
# Required for model-serving endpoints (refunder/complaint/support agents) to
# resolve UC functions at model-load time.  Without this, the first deploy of a
# new agent endpoint fails with `Permission denied: User does not have EXECUTE
# on Routine` because the endpoint's auto-generated SP can't traverse the
# catalog.  Granting here means every downstream stage inherits it.
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `account users`")

schemas_and_volumes = [
    ("food_safety",       "reports"),
    ("menu_documents",    "menus"),
    ("legal_complaints",  "documents"),
    ("regulatory",        "documents"),
    ("audits",            "reports"),
    ("consultancy",       "reports"),
]

for schema, volume in schemas_and_volumes:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.{volume}")
    print(f"\u2705 Created {CATALOG}.{schema}.{volume}")

##### Create prompt registry schema

A single Delta table holds versioned prompts for all agents in the demo.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.prompt_registry")
print(f"\u2705 Created schema {CATALOG}.prompt_registry")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.prompt_registry.prompts (
    name        STRING  NOT NULL COMMENT 'Unique prompt identifier, e.g. supervisor_system',
    version     INT     NOT NULL COMMENT 'Monotonically increasing version number',
    agent       STRING           COMMENT 'Agent or component this prompt belongs to',
    prompt      STRING  NOT NULL COMMENT 'Full prompt text',
    updated_at  TIMESTAMP        COMMENT 'Last modification timestamp',
    updated_by  STRING           COMMENT 'User or service principal that wrote this version'
)
USING DELTA
COMMENT 'Centralised prompt registry — one row per (name, version) pair'
""")
print(f"\u2705 Created table {CATALOG}.prompt_registry.prompts")

##### Upload pre-generated PDFs to Unity Catalog volumes

In [ ]:
import os
import glob

uploads = [
    ("../data/inspections/pdfs",      f"/Volumes/{CATALOG}/food_safety/reports"),
    ("../data/menus/pdfs",            f"/Volumes/{CATALOG}/menu_documents/menus"),
    ("../data/legal_complaints/pdfs", f"/Volumes/{CATALOG}/legal_complaints/documents"),
    ("../data/regulatory/pdfs",       f"/Volumes/{CATALOG}/regulatory/documents"),
    ("../data/audits/pdfs",           f"/Volumes/{CATALOG}/audits/reports"),
    ("../data/consultancy/pdfs",      f"/Volumes/{CATALOG}/consultancy/reports"),
]

for src_dir, vol_path in uploads:
    abs_src = os.path.abspath(src_dir)
    pdf_files = glob.glob(os.path.join(abs_src, "*.pdf"))
    print(f"Uploading {len(pdf_files)} PDFs from {src_dir} to {vol_path}")
    for pdf_file in pdf_files:
        filename = os.path.basename(pdf_file)
        with open(pdf_file, "rb") as src:
            with open(f"{vol_path}/{filename}", "wb") as dst:
                dst.write(src.read())
    print(f"\u2705 Uploaded {len(pdf_files)} PDFs to {vol_path}")

##### Load structured metadata as dimension tables

Each `*_metadata.json` file contains the source data used to generate the PDFs.
We load it once here so downstream stages can query it.

In [ ]:
import json
from datetime import date as dt_date
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType, DoubleType, ArrayType
)

##### Inspections + violations tables
with open(os.path.abspath("../data/inspections/inspection_metadata.json")) as f:
    inspection_meta = json.load(f)

print(
    f"Loaded metadata for {len(inspection_meta['inspections'])} inspections "
    f"across {len(inspection_meta['locations'])} locations"
)

inspection_rows = []
for insp in inspection_meta["inspections"]:
    inspection_rows.append({
        "inspection_id": insp["inspection_id"],
        "location_id": insp["location_id"],
        "location_name": insp["location_name"],
        "address": insp["address"],
        "jurisdiction": insp["jurisdiction"],
        "inspection_date": dt_date.fromisoformat(insp["inspection_date"]),
        "inspector_name": insp["inspector_name"],
        "score": insp["score"],
        "grade": insp["grade"],
        "violation_count": insp["violation_count"],
        "critical_count": insp["critical_count"],
        "major_count": insp["major_count"],
        "minor_count": insp["minor_count"],
        "follow_up_status": insp["follow_up_status"],
    })

inspection_schema = StructType([
    StructField("inspection_id", StringType()),
    StructField("location_id", IntegerType()),
    StructField("location_name", StringType()),
    StructField("address", StringType()),
    StructField("jurisdiction", StringType()),
    StructField("inspection_date", DateType()),
    StructField("inspector_name", StringType()),
    StructField("score", IntegerType()),
    StructField("grade", StringType()),
    StructField("violation_count", IntegerType()),
    StructField("critical_count", IntegerType()),
    StructField("major_count", IntegerType()),
    StructField("minor_count", IntegerType()),
    StructField("follow_up_status", StringType()),
])

df_inspections = spark.createDataFrame(inspection_rows, schema=inspection_schema)
df_inspections.write.mode("overwrite").saveAsTable(f"{CATALOG}.food_safety.inspections")
print(f"\u2705 Created inspections table with {df_inspections.count()} rows")

violation_rows = []
for insp in inspection_meta["inspections"]:
    for v in insp["violations"]:
        violation_rows.append({
            "inspection_id": insp["inspection_id"],
            "location_id": insp["location_id"],
            "location_name": insp["location_name"],
            "inspection_date": dt_date.fromisoformat(insp["inspection_date"]),
            "code": v["code"],
            "severity": v["severity"],
            "category": v["category"],
            "description": v["description"],
            "corrective_action": v["corrective_action"],
            "deadline_days": v["deadline_days"],
        })

violation_schema = StructType([
    StructField("inspection_id", StringType()),
    StructField("location_id", IntegerType()),
    StructField("location_name", StringType()),
    StructField("inspection_date", DateType()),
    StructField("code", StringType()),
    StructField("severity", StringType()),
    StructField("category", StringType()),
    StructField("description", StringType()),
    StructField("corrective_action", StringType()),
    StructField("deadline_days", IntegerType()),
])

df_violations = spark.createDataFrame(violation_rows, schema=violation_schema)
df_violations.write.mode("overwrite").saveAsTable(f"{CATALOG}.food_safety.violations")
print(f"\u2705 Created violations table with {df_violations.count()} rows")

In [ ]:
##### Menu brands metadata table
with open(os.path.abspath("../data/menus/menu_metadata.json")) as f:
    menu_meta = json.load(f)

print(f"Loaded menu metadata for {len(menu_meta['brands'])} brands")

menu_rows = []
for brand in menu_meta["brands"]:
    for item in brand["items"]:
        menu_rows.append({
            "brand_name": brand["brand_name"],
            "cuisine": brand["cuisine"],
            "pdf_filename": brand["pdf_filename"],
            "item_name": item["name"],
            "description": item["description"],
            "category": item["category"],
            "price": float(item["price"]),
            "calories": int(item["calories"]),
            "protein_g": int(item["protein_g"]),
            "fat_g": int(item["fat_g"]),
            "carbs_g": int(item["carbs_g"]),
            "allergens": item["allergens"],
        })

menu_schema = StructType([
    StructField("brand_name", StringType()),
    StructField("cuisine", StringType()),
    StructField("pdf_filename", StringType()),
    StructField("item_name", StringType()),
    StructField("description", StringType()),
    StructField("category", StringType()),
    StructField("price", DoubleType()),
    StructField("calories", IntegerType()),
    StructField("protein_g", IntegerType()),
    StructField("fat_g", IntegerType()),
    StructField("carbs_g", IntegerType()),
    StructField("allergens", ArrayType(StringType())),
])

df_menu = spark.createDataFrame(menu_rows, schema=menu_schema)
df_menu.write.mode("overwrite").saveAsTable(f"{CATALOG}.menu_documents.brands_metadata")
print(f"\u2705 Created menu_documents.brands_metadata table with {df_menu.count()} items")

##### Register stage completion with uc_state

Schemas and volumes are cleaned up when the catalog is dropped, so nothing
fine-grained needs to be tracked here — we just log the completion.

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add  # noqa: F401  (imported to keep parity with sibling stages)

print("\u2705 Document Data stage complete")